# Sort RER 2026 files from filenames only

This notebook has 3 phases:
1. Identify and remove likely non-student `proctor` / `debug` files (with review checkpoints).
2. Parse RER filenames.
3. Validate, preview destination paths, and copy into:

`sub-<student_id>/ses-<session_grade>_<true_season>/task-<task_folder>/`

Session grade `g0` from the filename is written as **KG** in the path (e.g. `ses-KG_fall`). Task codes `del`, `ble`, `nwr`, and `srt` map to **Deletion**, **Blending**, **NonwordRepetition**, and **SentenceRepetition**. Parsed CSV columns still keep the raw filename tokens (`grade`, `task`). Destination copies keep the original filename.

In [1]:
from pathlib import Path
import random
import re
import shutil

try:
    import pandas as pd
except ImportError:
    # Run this once in notebook if needed:
    # %pip install pandas
    raise ImportError('pandas is required for this notebook. Run: %pip install pandas')

In [2]:
# --- Configure paths ---
scores_root = Path('/Users/jeff/Documents/RER/LA_CZI_Phase2_unsorted/CZF_Media_20250916/Scores')
datalad_dataset = Path('/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted')

AUDIO_EXTENSIONS = {'.wav', '.mp3', '.m4a', '.flac'}

# Auto-discover all date-like folders (e.g., 20240923)
date_folders = sorted([
    p.name for p in scores_root.iterdir() if p.is_dir() and p.name.isdigit()
])

# Optional: narrow scope by including/excluding specific folders
include_only = []  # example: ['20240923', '20240930']
exclude_folders = []  # example: ['20231001']

if include_only:
    include_set = set(include_only)
    date_folders = [d for d in date_folders if d in include_set]
if exclude_folders:
    exclude_set = set(exclude_folders)
    date_folders = [d for d in date_folders if d not in exclude_set]

print(f'scores_root: {scores_root}')
print(f'found date_folders ({len(date_folders)}): {date_folders[:10]}')
if len(date_folders) > 10:
    print('... (showing first 10)')
print(f'datalad_dataset: {datalad_dataset}')

scores_root: /Users/jeff/Documents/RER/LA_CZI_Phase2_unsorted/CZF_Media_20250916/Scores
found date_folders (76): ['20240923', '20240924', '20241001', '20241002', '20241003', '20241004', '20241007', '20241008', '20241009', '20241015']
... (showing first 10)
datalad_dataset: /Users/jeff/Documents/RER/LA_CZI_Phase2_sorted


In [3]:
# --- Pre-clean helpers: proctor/debug detection ---
student_id_pattern = re.compile(r'\b\d{2}_\d{2}_\d{2}_\d{4}\b')
flag_terms = ('proctor', 'debug')


def collect_flagged_files(scores_root: Path, date_folders: list[str]) -> pd.DataFrame:
    rows = []

    for folder in date_folders:
        folder_path = scores_root / folder
        if not folder_path.exists():
            continue

        for p in folder_path.iterdir():
            if not p.is_file() or p.suffix.lower() not in AUDIO_EXTENSIONS:
                continue

            stem_lower = p.stem.lower()
            has_flag_term = any(term in stem_lower for term in flag_terms)
            if not has_flag_term:
                continue

            has_student_id = bool(student_id_pattern.search(p.stem))
            rows.append(
                {
                    'source_folder': folder,
                    'filename': p.name,
                    'source_path': str(p),
                    'has_student_id': has_student_id,
                    'delete_recommendation': 'review_keep' if has_student_id else 'safe_delete',
                }
            )

    if not rows:
        return pd.DataFrame(columns=['source_folder', 'filename', 'source_path', 'has_student_id', 'delete_recommendation'])
    return pd.DataFrame(rows)

In [4]:
# --- Preview proctor/debug files before deleting (compact review) ---
flagged_df = collect_flagged_files(scores_root, date_folders)

print(f'Flagged proctor/debug files: {len(flagged_df)}')
if len(flagged_df):
    flagged_df['matched_term'] = flagged_df['filename'].str.lower().apply(
        lambda x: 'debug' if 'debug' in x else ('proctor' if 'proctor' in x else 'other')
    )

    safe_delete_df = flagged_df[flagged_df['delete_recommendation'] == 'safe_delete'].copy()
    review_keep_df = flagged_df[flagged_df['delete_recommendation'] == 'review_keep'].copy()

    print('\nDelete recommendation counts:')
    print(flagged_df['delete_recommendation'].value_counts())

    print('\nCounts by folder (top 30):')
    folder_counts = (
        flagged_df.groupby('source_folder')
        .size()
        .sort_values(ascending=False)
        .reset_index(name='n_flagged')
    )
    display(folder_counts.head(30))

    print('\nCounts by matched term:')
    print(flagged_df['matched_term'].value_counts())

    print('\nSample of SAFE DELETE candidates (200 rows):')
    display(safe_delete_df[['source_folder', 'filename']].sample(min(200, len(safe_delete_df)), random_state=11))

    print('\nSample of NEEDS REVIEW candidates (up to 200 rows):')
    if len(review_keep_df):
        display(review_keep_df[['source_folder', 'filename']].sample(min(200, len(review_keep_df)), random_state=11))
    else:
        print('None')

    # Optional full review export
    review_csv = Path('flagged_proctor_debug_review.csv')
    flagged_df.sort_values(['source_folder', 'filename']).to_csv(review_csv, index=False)
    print(f'\nWrote full candidate list to: {review_csv.resolve()}')
else:
    safe_delete_df = pd.DataFrame(columns=flagged_df.columns)
    review_keep_df = pd.DataFrame(columns=flagged_df.columns)

Flagged proctor/debug files: 0


In [5]:
# --- Delete flagged files + remove empty folders (compact dry-run summary) ---
def delete_files_and_empty_dirs(
    paths: list[Path],
    dry_run: bool = True,
    preview_n: int = 30,
    write_manifest: bool = True,
) -> tuple[int, int]:
    deleted_files = 0
    deleted_dirs = 0
    touched_dirs = set()

    for p in paths:
        touched_dirs.add(p.parent)

    if dry_run:
        print(f'Dry run: {len(paths)} files would be deleted.')
        preview_paths = paths[:preview_n]
        print(f'Previewing first {len(preview_paths)} file deletions:')
        for p in preview_paths:
            print(f'  [DRY RUN] {p}')

        empty_dirs = [d for d in sorted(touched_dirs) if d.exists() and d.is_dir() and not any(d.iterdir())]
        print(f'Potential empty source folders after deletion: {len(empty_dirs)}')
        if empty_dirs:
            print('Previewing first 20 empty folders:')
            for d in empty_dirs[:20]:
                print(f'  [DRY RUN] {d}')

        if write_manifest:
            manifest = Path('safe_delete_manifest.txt')
            manifest.write_text('\n'.join(str(p) for p in paths) + '\n')
            print(f'Wrote delete manifest: {manifest.resolve()}')

        print('\nDry run complete. No files/folders were deleted.')
        return 0, 0

    for p in paths:
        if p.exists() and p.is_file():
            p.unlink()
            deleted_files += 1

    for d in sorted(touched_dirs):
        if d.exists() and d.is_dir() and not any(d.iterdir()):
            d.rmdir()
            deleted_dirs += 1

    print(f'\nDeleted files: {deleted_files}')
    print(f'Deleted empty folders: {deleted_dirs}')
    return deleted_files, deleted_dirs

# Step 1: dry run on recommended safe deletes
safe_delete_paths = [Path(p) for p in safe_delete_df['source_path'].tolist()]
delete_files_and_empty_dirs(safe_delete_paths, dry_run=True, preview_n=30)

# Step 2: actual delete (uncomment after review)
# delete_files_and_empty_dirs(safe_delete_paths, dry_run=False)

Dry run: 0 files would be deleted.
Previewing first 0 file deletions:
Potential empty source folders after deletion: 0
Wrote delete manifest: /Users/jeff/Documents/rer_sort_code/safe_delete_manifest.txt

Dry run complete. No files/folders were deleted.


(0, 0)

In [6]:
# --- Filename parser ---
season_map = {
    'f': 'fall',
    'w': 'winter',
    's': 'spring',
    'e': 'end',
}


def parse_rer_filename(filename: str) -> dict:
    """
    Expected pattern:
    grade_assessmentperiod_year_sid1_sid2_sid3_sid4_task_item..._audioid.ext
    """
    p = Path(filename)
    stem = p.stem
    tokens = stem.split('_')

    result = {
        'filename': p.name,
        'grade': None,
        'assessment_period': None,
        'true_season': None,
        'year': None,
        'student_id': None,
        'task': None,
        'item': None,
        'audio_id': None,
        'parse_ok': False,
        'parse_error': None,
    }

    if 'debug-' in stem.lower():
        result['parse_error'] = 'debug_filename'
        return result

    if len(tokens) < 10:
        result['parse_error'] = f'too_few_tokens:{len(tokens)}'
        return result

    grade, assessment_period, year = tokens[0], tokens[1], tokens[2]
    sid_tokens = tokens[3:7]
    task = tokens[7]
    audio_id = tokens[-1]
    item_tokens = tokens[8:-1]

    if not item_tokens:
        result['parse_error'] = 'missing_item'
        return result

    result.update(
        {
            'grade': grade,
            'assessment_period': assessment_period,
            'true_season': season_map.get(assessment_period.lower(), 'unknown'),
            'year': year,
            'student_id': '_'.join(sid_tokens),
            'task': task,
            'item': '_'.join(item_tokens),
            'audio_id': audio_id,
        }
    )

    sid_ok = bool(re.fullmatch(r'\d{2}_\d{2}_\d{2}_\d{4}', result['student_id']))
    grade_ok = bool(re.fullmatch(r'g\d+', grade.lower()))

    if not sid_ok:
        result['parse_error'] = 'bad_student_id_pattern'
        return result
    if not grade_ok:
        result['parse_error'] = 'bad_grade_pattern'
        return result

    result['parse_ok'] = True
    return result

In [7]:
# --- Collect files and parse ---
def collect_parsed_rows(scores_root: Path, date_folders: list[str]) -> pd.DataFrame:
    rows = []

    for folder in date_folders:
        folder_path = scores_root / folder
        if not folder_path.exists():
            # print(f'skipping missing folder: {folder_path}')
            continue

        files = [p for p in folder_path.iterdir() if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS]
        print(f'{folder}: {len(files)} audio files')

        for p in files:
            parsed = parse_rer_filename(p.name)
            parsed['source_folder'] = folder
            parsed['source_path'] = str(p)
            rows.append(parsed)

    return pd.DataFrame(rows)


def sample_rows_by_folder(df: pd.DataFrame, n: int = 5, seed: int = 42) -> pd.DataFrame:
    random.seed(seed)
    samples = []
    for folder, group in df.groupby('source_folder'):
        take_n = min(n, len(group))
        samples.append(group.sample(n=take_n, random_state=seed))
    if not samples:
        return pd.DataFrame()
    return pd.concat(samples).sort_values(['source_folder', 'filename'])

In [8]:
df = collect_parsed_rows(scores_root, date_folders)
print(f'\nTotal parsed rows: {len(df)}')

if len(df):
    print('\nParse status counts:')
    print(df['parse_ok'].value_counts(dropna=False))

    print('\nUnique assessment_period -> true_season pairs (left column is NOT a count):')
    print(
        df[['assessment_period', 'true_season']]
        .drop_duplicates()
        .sort_values('assessment_period')
        .reset_index(drop=True)
    )
    print('\nFiles per assessment_period:')
    print(df['assessment_period'].value_counts().sort_index())

    print('\nUnique tasks (first 30):')
    print(sorted(df['task'].dropna().unique())[:30])

20240923: 937 audio files
20240924: 1630 audio files
20241001: 1033 audio files
20241002: 1788 audio files
20241003: 2326 audio files
20241004: 1842 audio files
20241007: 1167 audio files
20241008: 816 audio files
20241009: 59 audio files
20241015: 1724 audio files
20241016: 1422 audio files
20241017: 961 audio files
20241018: 1010 audio files
20241021: 1041 audio files
20241022: 1238 audio files
20241023: 1230 audio files
20241024: 1278 audio files
20241025: 346 audio files
20241028: 562 audio files
20241029: 1497 audio files
20241030: 1547 audio files
20241031: 920 audio files
20241101: 709 audio files
20250113: 773 audio files
20250114: 1563 audio files
20250115: 1240 audio files
20250116: 716 audio files
20250117: 131 audio files
20250121: 1015 audio files
20250122: 1154 audio files
20250123: 558 audio files
20250124: 66 audio files
20250128: 2823 audio files
20250129: 3338 audio files
20250130: 3186 audio files
20250131: 2452 audio files
20250203: 1071 audio files
20250204: 2403 a

In [9]:
# --- Validation checkpoints + CSV exports for deep review ---
bad_df = df[~df['parse_ok']].copy()
ok_review_df = df[df['parse_ok']].copy()

print(f'Bad parse rows: {len(bad_df)}')
if len(bad_df):
    display_cols = ['source_folder', 'filename', 'parse_error']
    display(bad_df[display_cols].head(50))

check_cols = [
    'source_folder',
    'filename',
    'grade',
    'student_id',
    'assessment_period',
    'true_season',
    'task',
    'item',
    'audio_id',
    'source_path',
]

# Quick in-notebook spot-check
print('\nRandom sample per folder for manual spot-check (in notebook):')
sample_df = sample_rows_by_folder(ok_review_df, n=8, seed=7)
display(sample_df[check_cols])

# CSV exports for full audit in spreadsheet software
review_dir = Path('review_exports')
review_dir.mkdir(exist_ok=True)

full_ok_csv = review_dir / 'parsed_ok_all_rows.csv'
bad_csv = review_dir / 'parsed_bad_rows.csv'

# balanced sample: at least some from each folder
per_folder_sample_n = 30  # increase if you want deeper per-folder sampling
sample_per_folder_df = sample_rows_by_folder(ok_review_df, n=per_folder_sample_n, seed=17)
sample_csv = review_dir / f'parsed_ok_sample_{per_folder_sample_n}_per_folder.csv'

ok_review_df[check_cols].sort_values(['source_folder', 'filename']).to_csv(full_ok_csv, index=False)
bad_df[['source_folder', 'filename', 'parse_error', 'source_path']].sort_values(['source_folder', 'filename']).to_csv(bad_csv, index=False)
sample_per_folder_df[check_cols].to_csv(sample_csv, index=False)

print('\nWrote review CSVs:')
print(f'- Full parsed OK rows: {full_ok_csv.resolve()}')
print(f'- Bad parse rows: {bad_csv.resolve()}')
print(f'- Sample per folder: {sample_csv.resolve()}')
print(f'  sample rows: {len(sample_per_folder_df)} (up to {per_folder_sample_n} per folder)')

Bad parse rows: 0

Random sample per folder for manual spot-check (in notebook):


,source_folder,filename,grade,student_id,assessment_period,true_season,task,item,audio_id,source_path
493,20240923,g0_f_24_11_01_24_0058_del_motel_765074.wav,g0,11_01_24_0058,f,fall,del,motel,765074,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...
729,20240923,g0_f_24_11_01_24_0061_del_hs_p_baseball_2_7B5B...,g0,11_01_24_0061,f,fall,del,hs_p_baseball_2,7B5B08,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...
259,20240923,g0_f_24_11_01_24_0061_syn_hs_p_little_FEC75E.wav,g0,11_01_24_0061,f,fall,syn,hs_p_little,FEC75E,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...
871,20240923,g0_f_24_11_01_24_0065_del_mice_9CC943.wav,g0,11_01_24_0065,f,fall,del,mice,9CC943,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...
572,20240923,g0_f_24_11_01_24_0067_del_cup_9D66AE.wav,g0,11_01_24_0067,f,fall,del,cup,9D66AE,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...
...,...,...,...,...,...,...,...,...,...,...
123505,20250502,g0_s_24_13_01_24_0275_wj4_lwid_special_412A48.wav,g0,13_01_24_0275,s,spring,wj4,lwid_special,412A48,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...
123432,20250502,g0_s_24_13_03_24_0497_wj4_lwid_k_DD28DD.wav,g0,13_03_24_0497,s,spring,wj4,lwid_k,DD28DD,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...
123469,20250502,g0_s_24_13_03_24_0497_wj4_lwid_often_E5CF03.wav,g0,13_03_24_0497,s,spring,wj4,lwid_often,E5CF03,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...
123444,20250502,g0_s_24_13_03_24_0506_wj4_wa_cl_293DB5.wav,g0,13_03_24_0506,s,spring,wj4,wa_cl,293DB5,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...



Wrote review CSVs:
- Full parsed OK rows: /Users/jeff/Documents/rer_sort_code/review_exports/parsed_ok_all_rows.csv
- Bad parse rows: /Users/jeff/Documents/rer_sort_code/review_exports/parsed_bad_rows.csv
- Sample per folder: /Users/jeff/Documents/rer_sort_code/review_exports/parsed_ok_sample_30_per_folder.csv
  sample rows: 2280 (up to 30 per folder)


In [10]:
# --- Build destination paths ---
# Folder labels (filename tokens stay as g0, del, etc. in review CSVs)
TASK_FOLDER_NAMES = {
    'del': 'Deletion',
    'ble': 'Blending',
    'nwr': 'NonwordRepetition',
    'srt': 'SentenceRepetition',
}


def ses_grade_label(grade: str) -> str:
    """Map parsed grade token to session folder segment (g0 -> KG)."""
    return 'KG' if grade.lower() == 'g0' else grade.lower()


def task_folder_label(task: str) -> str:
    return TASK_FOLDER_NAMES.get(task.lower(), task)


def build_destination(row: pd.Series, datalad_dataset: Path) -> Path:
    g = ses_grade_label(row['grade'])
    t = task_folder_label(row['task'])
    sub_ses_task = (
        datalad_dataset
        / f"sub-{row['student_id']}"
        / f"ses-{g}_{row['true_season']}"
        / f"task-{t}"
    )
    return sub_ses_task / row['filename']

ok_df = df[df['parse_ok']].copy()
ok_df['dest_path'] = ok_df.apply(lambda r: str(build_destination(r, datalad_dataset)), axis=1)

print(f'Rows ready to copy: {len(ok_df)}')
print('\nSample destination mappings:')
display(ok_df[['source_path', 'dest_path']].sample(min(20, len(ok_df)), random_state=10))

print('\nTop destination task folders by file count:')
preview_counts = (
    ok_df.assign(task_folder=lambda d: d['dest_path'].str.rsplit('/', n=1).str[0])
    .groupby('task_folder')
    .size()
    .sort_values(ascending=False)
)
print(preview_counts.head(20))

Rows ready to copy: 123514

Sample destination mappings:


,source_path,dest_path
32140,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...,/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted...
110181,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...,/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted...
16071,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...,/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted...
122407,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...,/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted...
69018,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...,/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted...
63938,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...,/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted...
55376,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...,/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted...
73221,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...,/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted...
59518,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...,/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted...
18168,/Users/jeff/Documents/RER/LA_CZI_Phase2_unsort...,/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted...



Top destination task folders by file count:
task_folder
/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted/sub-13_03_24_0478/ses-KG_spring/task-wj4    172
/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted/sub-13_01_24_0342/ses-KG_spring/task-wj4    167
/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted/sub-13_01_24_0361/ses-KG_spring/task-wj4    153
/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted/sub-11_03_24_0242/ses-KG_spring/task-wj4    147
/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted/sub-11_01_24_0011/ses-KG_spring/task-wj4    119
/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted/sub-13_03_24_0496/ses-KG_spring/task-wj4    118
/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted/sub-12_01_24_0164/ses-KG_spring/task-wj4    118
/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted/sub-13_03_24_0497/ses-KG_spring/task-wj4    116
/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted/sub-13_03_24_0471/ses-KG_spring/task-wj4    113
/Users/jeff/Documents/RER/LA_CZI_Phase2_sorted/sub-12_01_24_0195/ses-KG_spring/task-wj4 

In [11]:
# --- Copy operation (dry-run first) ---
def copy_files(df_to_copy: pd.DataFrame, dry_run: bool = True, max_files: int | None = None):
    copied = 0
    rows = df_to_copy if max_files is None else df_to_copy.head(max_files)

    for _, row in rows.iterrows():
        src = Path(row['source_path'])
        dst = Path(row['dest_path'])

        if dry_run:
            print(f'[DRY RUN] {src} -> {dst}')
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
        copied += 1

    mode = 'would copy' if dry_run else 'copied'
    print(f'\nDone: {mode} {copied} files.')

In [14]:
# Step 1: dry-run a small subset
# copy_files(ok_df, dry_run=True, max_files=25)

# Step 2: dry-run everything (uncomment when ready)
# copy_files(ok_df, dry_run=True)

# Step 3: actual copy (uncomment only after validation)
copy_files(ok_df, dry_run=False)


Done: copied 123514 files.


In [ ]:
import numpy as np
# --- Post-copy validation (run after ok_df + copy) ---
# Checks: catalog vs path layout, no duplicate destinations, every dest exists and matches source size.

import filecmp


def _expected_path_parts(df: pd.DataFrame) -> pd.DataFrame:
    g = df["grade"].str.lower()
    ses_grade = np.where(g == "g0", "KG", g)
    tl = df["task"].str.lower()
    mapped = tl.map(TASK_FOLDER_NAMES)
    task_folder = mapped.combine_first(df["task"])
    out = df.copy()
    out["_exp_sub"] = "sub-" + out["student_id"].astype(str)
    out["_exp_ses"] = "ses-" + ses_grade.astype(str) + "_" + out["true_season"].astype(str)
    out["_exp_task"] = "task-" + task_folder.astype(str)
    out["_exp_name"] = out["filename"].astype(str)
    return out


def validate_catalog_paths(df: pd.DataFrame) -> pd.DataFrame:
    """Rows where dest_path parent chain does not match parsed fields + naming rules."""
    x = _expected_path_parts(df)
    d = x["dest_path"].map(Path)
    ok = (
        d.map(lambda p: p.name) == x["_exp_name"]
    ) & (
        d.map(lambda p: p.parent.name) == x["_exp_task"]
    ) & (
        d.map(lambda p: p.parent.parent.name) == x["_exp_ses"]
    ) & (
        d.map(lambda p: p.parent.parent.parent.name) == x["_exp_sub"]
    )
    return x.loc[~ok]


def validate_duplicates(df: pd.DataFrame) -> pd.Series:
    dup = df["dest_path"].duplicated(keep=False)
    return df.loc[dup, "dest_path"]


def validate_on_disk(df: pd.DataFrame, binary_cmp_sample: int = 300) -> dict:
    """Every dest exists and matches source byte size; full filecmp on a random sample."""
    rng = random.Random(42)
    n_bad_exist, n_bad_size = 0, 0
    bad_exist, bad_size = [], []

    srcs = df["source_path"].tolist()
    dsts = df["dest_path"].tolist()
    for sp, dp in zip(srcs, dsts):
        src, dst = Path(sp), Path(dp)
        if not dst.is_file():
            n_bad_exist += 1
            if len(bad_exist) < 15:
                bad_exist.append((str(src), str(dst)))
            continue
        if src.is_file() and src.stat().st_size != dst.stat().st_size:
            n_bad_size += 1
            if len(bad_size) < 15:
                bad_size.append((str(src), str(dst), src.stat().st_size, dst.stat().st_size))

    idx = list(range(len(df)))
    rng.shuffle(idx)
    cmp_fail = []
    for i in idx[: min(binary_cmp_sample, len(df))]:
        src, dst = Path(srcs[i]), Path(dsts[i])
        if not src.is_file() or not dst.is_file():
            continue
        if not filecmp.cmp(src, dst, shallow=False):
            cmp_fail.append((str(src), str(dst)))
            if len(cmp_fail) >= 10:
                break

    return {
        "n_rows": len(df),
        "n_bad_exist": n_bad_exist,
        "n_bad_size": n_bad_size,
        "sample_bad_exist": bad_exist,
        "sample_bad_size": bad_size,
        "binary_cmp_sample_n": min(binary_cmp_sample, len(df)),
        "binary_cmp_failures_shown": cmp_fail,
    }


print("=== Post-copy validation ===\n")

_n = min(2000, len(ok_df))
_samp = ok_df.sample(n=_n, random_state=3) if _n else ok_df
_rebuilt = _samp.apply(lambda r: str(build_destination(r, datalad_dataset)), axis=1)
_bm = int((_rebuilt.values != _samp["dest_path"].values).sum())
print(f"0) build_destination() vs stored dest_path (random n={_n}): {_bm} mismatches (expected 0)")
if _bm:
    display(_samp.assign(rebuilt=_rebuilt).loc[_rebuilt.values != _samp["dest_path"].values, ["source_path", "dest_path", "rebuilt"]].head(15))

miss = validate_catalog_paths(ok_df)
print(f"1) Path layout vs catalog fields: {len(miss)} mismatches (expected 0)")
if len(miss):
    display(miss[["source_path", "dest_path", "grade", "true_season", "task", "student_id", "filename"]].head(20))

dups = validate_duplicates(ok_df)
print(f"\n2) Duplicate dest_path rows: {len(dups)} (expected 0)")
if len(dups):
    display(ok_df[ok_df["dest_path"].duplicated(keep=False)].sort_values("dest_path").head(30))

disk = validate_on_disk(ok_df)
print(f"\n3) On disk: missing dest files: {disk['n_bad_exist']}; size mismatch vs source: {disk['n_bad_size']}")
if disk["sample_bad_exist"]:
    print("  sample missing:", disk["sample_bad_exist"][:3])
if disk["sample_bad_size"]:
    print("  sample size mismatch:", disk["sample_bad_size"][:3])

print(f"\n4) Binary compare (first {disk['binary_cmp_sample_n']} random rows after shuffle seed=42): {len(disk['binary_cmp_failures_shown'])} failures shown")
if disk["binary_cmp_failures_shown"]:
    for a, b in disk["binary_cmp_failures_shown"]:
        print("  ", a, "->", b)

all_ok = (
    _bm == 0
    and len(miss) == 0
    and len(dups) == 0
    and disk["n_bad_exist"] == 0
    and disk["n_bad_size"] == 0
    and len(disk["binary_cmp_failures_shown"]) == 0
)
print("\n=== Overall:", "PASS" if all_ok else "FAIL — inspect blocks above", "===")


### ReadNet audio counts (for reconciliation)

Exports **`review_exports/audio_counts_by_semester_task_grade.csv`**: one row per **semester** × **task** × **grade**. **semester** is `true_season` (fall / spring / winter). **task** and **grade** match ReadNet folder names (`Deletion`, `KG`, …). **`reconcile_key`** is `semester_task_grade` joined with underscores for easy matching in another sheet. **`n_audio_files`** counts rows in `ok_df` (same files as under `sub-*/ses-<grade>_<semester>/task-*/`).

Re-run this cell after refreshing `ok_df` if you re-copy or change scope.


In [16]:
# --- Spreadsheet: audio counts by semester × task × grade (ReadNet layout) ---
# Requires: ok_df + TASK_FOLDER_NAMES from the build-dest cell.

import numpy as np

review_dir = Path("review_exports")
review_dir.mkdir(exist_ok=True)

_g = ok_df["grade"].str.lower()
report_grade = np.where(_g == "g0", "KG", _g)
_tl = ok_df["task"].str.lower()
_mapped = _tl.map(TASK_FOLDER_NAMES)
report_task = _mapped.combine_first(ok_df["task"])
report_semester = ok_df["true_season"].astype(str)

audio_counts_by_semester_task_grade = (
    ok_df.assign(
        semester=report_semester,
        task=report_task,
        grade=report_grade,
    )
    .groupby(["semester", "task", "grade"], as_index=False)
    .size()
    .rename(columns={"size": "n_audio_files"})
    .assign(
        reconcile_key=lambda d: d["semester"].astype(str)
        + "_"
        + d["task"].astype(str)
        + "_"
        + d["grade"].astype(str)
    )
    .sort_values(["semester", "grade", "task"])
    .reset_index(drop=True)
)

audio_counts_by_semester_task_grade = audio_counts_by_semester_task_grade[
    ["reconcile_key", "semester", "task", "grade", "n_audio_files"]
]

out_csv = review_dir / "audio_counts_by_semester_task_grade.csv"
audio_counts_by_semester_task_grade.to_csv(out_csv, index=False)
print(f"Wrote: {out_csv.resolve()}")
print(f"Unique combinations: {len(audio_counts_by_semester_task_grade)}")
print(f"Total files (sum of n_audio_files): {int(audio_counts_by_semester_task_grade['n_audio_files'].sum())}  (ok_df rows: {len(ok_df)})")
display(audio_counts_by_semester_task_grade.head(40))

# Optional Excel (uncomment if openpyxl is installed)
# out_xlsx = review_dir / "audio_counts_by_semester_task_grade.xlsx"
# audio_counts_by_semester_task_grade.to_excel(out_xlsx, index=False)
# print(f"Wrote: {out_xlsx.resolve()}")


Wrote: /Users/jeff/Documents/rer_sort_code/review_exports/audio_counts_by_semester_task_grade.csv
Unique combinations: 31
Total files (sum of n_audio_files): 123514  (ok_df rows: 123514)


,reconcile_key,semester,task,grade,n_audio_files
0,fall_Blending_KG,fall,Blending,KG,4359
1,fall_Deletion_KG,fall,Deletion,KG,6125
2,fall_SentenceRepetition_KG,fall,SentenceRepetition,KG,4676
3,fall_evo_KG,fall,evo,KG,4214
4,fall_gms_KG,fall,gms,KG,470
5,fall_lnc_KG,fall,lnc,KG,573
6,fall_ron_KG,fall,ron,KG,1307
7,fall_syn_KG,fall,syn,KG,4896
8,fall_upper_KG,fall,upper,KG,463
9,spring_Deletion_KG,spring,Deletion,KG,2838
